In [0]:
%sql
INSERT INTO catalog_ventas.gold.dim_fecha (
    id_fecha,
    vtafecha,
    anio,
    mes,
    trimestre,
    dia_semana,
    es_fin_de_semana,
    _created_at
)
SELECT 
    CAST(DATE_FORMAT(vtafecha, 'yyyyMMdd') AS BIGINT) AS id_fecha,
    vtafecha,
    YEAR(vtafecha),
    MONTH(vtafecha),
    QUARTER(vtafecha),
    CASE DAYOFWEEK(vtafecha)
        WHEN 1 THEN 'Domingo'
        WHEN 2 THEN 'Lunes'
        WHEN 3 THEN 'Martes'
        WHEN 4 THEN 'Miercoles'
        WHEN 5 THEN 'Jueves'
        WHEN 6 THEN 'Viernes'
        WHEN 7 THEN 'Sabado'
    END,
    CASE WHEN DAYOFWEEK(vtafecha) IN (1,7) THEN TRUE ELSE FALSE END,
    CURRENT_TIMESTAMP()
FROM catalog_ventas.silver.ventas_clean src

WHERE vtafecha IS NOT NULL
AND NOT EXISTS (
    SELECT 1
    FROM catalog_ventas.gold.dim_fecha tgt
    WHERE tgt.id_fecha = CAST(DATE_FORMAT(src.vtafecha, 'yyyyMMdd') AS BIGINT)
);


In [0]:
%sql
describe catalog_ventas.gold.dim_fecha 

In [0]:
%sql
INSERT INTO catalog_ventas.gold.dim_fecha (
    id_fecha,
    vtafecha,
    anio,
    mes,
    trimestre,
    dia_semana,
    es_fin_de_semana,
    _created_at
)
SELECT
    CAST(DATE_FORMAT(CAST(vtafecha AS DATE), 'yyyyMMdd') AS BIGINT) AS id_fecha,
    CAST(vtafecha AS DATE) AS vtafecha,
    YEAR(CAST(vtafecha AS DATE)),
    MONTH(CAST(vtafecha AS DATE)),
    QUARTER(CAST(vtafecha AS DATE)),
    CASE DAYOFWEEK(CAST(vtafecha AS DATE))
        WHEN 1 THEN 'Domingo'
        WHEN 2 THEN 'Lunes'
        WHEN 3 THEN 'Martes'
        WHEN 4 THEN 'Miercoles'
        WHEN 5 THEN 'Jueves'
        WHEN 6 THEN 'Viernes'
        WHEN 7 THEN 'Sabado'
    END,
    CASE WHEN DAYOFWEEK(CAST(vtafecha AS DATE)) IN (1,7) THEN TRUE ELSE FALSE END,
    CURRENT_TIMESTAMP()
FROM (
    SELECT DISTINCT CAST(vtafecha AS DATE) AS vtafecha
    FROM catalog_ventas.silver.ventas_clean
    WHERE vtafecha IS NOT NULL
) src
WHERE NOT EXISTS (
    SELECT 1
    FROM catalog_ventas.gold.dim_fecha tgt
    WHERE tgt.id_fecha = CAST(DATE_FORMAT(src.vtafecha, 'yyyyMMdd') AS BIGINT)
);